# Figure 7 ver9 + 보충 그림 1–4 검산 (2026-11-18)
ver9는 그림 재구성판입니다. 통계는 ver8 CSV(2026-11-17 검산 완료, 이후 변경 없음)와 공개 코호트 v8.1 CSV를 읽습니다. 이 노트북은 ① ver9에서 새로 등장한 수치(코호트 구성, 채혈 수, 사망 확장, n80 문장)를 원본 메타데이터로 다시 계산하고, ② Results의 공개 코호트 수치를 v8.1 CSV와 대조하며, ③ 공개 코호트 독립 재구현(v8.1)을 실행합니다. FAIL은 원본 `Results_and_Legends_ver9.md` 또는 그림 라벨의 오류입니다.

환경 변수: `PBISC_ROOT` (`eskd_covid_clinical_assoc_20261115/`, `paper_claim_analysis/`, `public_covid_who_hspc_nk_20261115/`, `HSPC_CD56brightNK_COVID_analysis_summary_20261116/`이 있는 폴더), `PUB_INDEP` (독립 재구현 스크립트 경로).

In [ ]:
import os, numpy as np, pandas as pd
ROOT = os.environ.get('PBISC_ROOT', r'F:\PBISC'); SUM = os.path.join(ROOT, 'HSPC_CD56brightNK_COVID_analysis_summary_20261116')
S = pd.read_csv(os.path.join(ROOT, 'eskd_covid_clinical_assoc_20261115', 'results', 'sample_level_composition_metadata.csv'))
RC = pd.read_csv(os.path.join(ROOT, 'paper_claim_analysis', 'covid_sample_patient_id_reconciliation.csv'))
V8 = os.path.join(SUM, '15_Figure7_ver8_Main_and_Supp', 'data'); PUB = os.path.join(SUM, '16_Public_COVID_Replication_ver8', 'data')
issues = []
def check(label, ok, detail=''):
    print(('PASS ' if ok else 'FAIL ') + label + ('' if ok else f'  -> {detail}'))
    if not ok: issues.append(label)

## 1. 코호트 구성 (Fig. 7a–b, Supp. Fig. 1)

In [ ]:
CLL = {'C116','C139'}
d = S[(S.condition=='positive') & S.in_primary_scope.astype(bool) & S.qc_pass_min200cells.astype(bool) & ~S.patient_id.isin(CLL)]
R = d[d.source=='Real']; B = d[d.source=='PBISC']; Rp = set(R.patient_id); N = B[~B.patient_id.isin(Rp)]; SH = Rp & set(B.patient_id)
check('Real 37 pts / 138 draws; PBISC new 29 / 79; 217 total; shared 27; Real-only 10',
      (len(Rp), len(R), N.patient_id.nunique(), len(N), len(R)+len(N), len(SH), len(Rp-SH)) == (37,138,29,79,217,27,10))
ro = R[R.patient_id.isin(Rp-SH)].groupby('patient_id').centre_or_wave.first().value_counts().to_dict()
check('Fig. 7a label: Real-only = "Wave 2"', False, f'Real-only by centre: {ro}')
gR, gN = R.groupby('patient_id'), N.groupby('patient_id')
w = pd.DataFrame({'Real': gR.severity_ord.first().value_counts(), 'New': gN.severity_ord.first().value_counts()}).sort_index(); print(w)
check('WHO 3: 5 -> 12 (+140%)', (w.loc[3.0,'Real'], w.loc[3.0,'Real']+w.loc[3.0,'New']) == (5,12))
check('deaths 7 -> 13', (int(gR.fatal_harmonized.first().sum()), int(gR.fatal_harmonized.first().sum()+gN.fatal_harmonized.first().sum())) == (7,13))
check('PBISC new all w1; Real NCL 21 / Cambridge 16', gN.centre_or_wave.first().eq('w1').all() and gR.centre_or_wave.first().value_counts().to_dict()=={'NCL':21,'Cambridge':16})
st = RC.set_index('sample_id').published_received_steroid.eq('Y'); ster = R.sample_id.astype(str).map(st).fillna(False)
check('steroid-exposed Real: 7 pts / 34 draws', (int(ster.groupby(R.patient_id).max().sum()), int(ster.sum())) == (7,34))
ncl = R[R.centre_or_wave=='NCL']; check('Supp. Fig. 1c: "17/88 NCL draws <2% mono"', (int((ncl.Mono_pct<2).sum()), len(ncl)) == (17,88), f'{int((ncl.Mono_pct<2).sum())}/{len(ncl)} in the analysed COVID-19 cohort')
dpp = pd.concat([gR.size(), gN.size()]).clip(upper=4).value_counts().sort_index().to_dict()
check('draws/patient 1:8, 2:10, 3:26, >=4:22', dpp == {1:8, 2:10, 3:26, 4:22}, dpp)
check('sampling span Real 0-20 (median 9), New 0-30 (median 12)', (R.days_from_first_positive_swab.max(), R.days_from_first_positive_swab.median(), N.days_from_first_positive_swab.max(), N.days_from_first_positive_swab.median()) == (20,9,30,12))
age = pd.concat([gR.age.first(), gN.age.first()]); print('median age, 66 analysed patients:', age.median())
raw = pd.read_csv(os.path.join(V8, 'patient_raw_pct_noCLL_v8.csv'), index_col=0)
print('CD14mono_dex % (steroid marker) by source:'); print(raw.groupby('src').CD14mono_dex.describe().round(3))

## 2. n80 문장 (Fig. 7f, q)

In [ ]:
PW = pd.read_csv(os.path.join(V8, 'power_curves_v8.csv'))
def n80(g, col):
    g = g.sort_values('n_real'); x, y = g.n_real.values, g[col].values
    if y[0] >= .8: return '<=10'
    for i in range(1, len(x)):
        if y[i] >= .8: return round(x[i-1] + (.8-y[i-1])*(x[i]-x[i-1])/(y[i]-y[i-1]))
    return 'n.r.'
T = pd.DataFrame([(f, o, n80(g,'real_alone'), n80(g,'plus_pbisc_new')) for (f,o),g in PW.groupby(['feature','outcome'], sort=False)], columns=['feature','outcome','Real','+PBISC']); print(T)
anch = T[T.feature.isin(['NK CD56 bright','B_ASC_dividing','CD16mono']) | ((T.feature=='T CD8 CM')&(T.outcome=='sev'))]
check('anchors Real-alone n80 = 19–24', sorted(anch.Real) == [19,20,21,24], list(anch.Real))
foc = T[T.feature.isin(['B_ASC_IgG','B_ASC_IgA','HSPC']) | ((T.feature=='T CD8 CTL')) | ((T.feature=='T CD8 CM')&(T.outcome=='death'))]
real_nums = [v for v in foc.Real if v != 'n.r.']
check('Results: followed Real-alone n80 = 26 to n.r.', min(real_nums) == 26, f'min Real-alone n80 = {min(real_nums)} (rest n.r.)')
check('Results/Fig. 7f text: PBISC new reduces all to <=10', (foc['+PBISC'] == '<=10').all(), foc[foc['+PBISC'] != '<=10'][['feature','outcome','+PBISC']].values.tolist())

## 3. Results의 공개 코호트 수치 (v8.1, Supp. Fig. 4)

In [ ]:
M = pd.read_csv(os.path.join(PUB, 'public_meta_analysis_v8.csv')).set_index('feature'); Cc = pd.read_csv(os.path.join(PUB, 'public_cohort_level_results_v8.csv')).set_index(['cohort','feature'])
G = pd.read_csv(os.path.join(PUB, 'stephenson_b_asc_isotype_glm_v8.csv')).set_index('feature'); H = pd.read_csv(os.path.join(PUB, 'public_hspc_timewindow_death_v8.csv')).set_index(['cohort','window'])
r2 = lambda x: round(float(x), 2)
for f, rho, p in [('NK_CD56bright',-0.25,0.0014),('B_ASC_dividing',0.33,5.2e-4),('T_CD8_CM',-0.33,0.0026),('CD16_mono_ofMono',-0.35,0.0093),('B_ASC_total',0.28,0.0028),('HSPC',0.45,2.8e-8)]:
    check(f'meta {f}: rho {rho:+.2f}, P {p:g}', r2(M.loc[f,'rho_meta'])==rho and np.isclose(M.loc[f,'p_rho_meta'], p, rtol=0.06), (M.loc[f,'rho_meta'], M.loc[f,'p_rho_meta']))
check('NK CD56bright / B ASC total now k=4, N=219', [int(M.loc[f,'k_cohorts']) for f in ['NK_CD56bright','B_ASC_total']]==[4,4] and int(M.loc['B_ASC_total','N_patients'])==219)
check('B ASC total death AUC 0.46 P 0.77', (r2(M.loc['B_ASC_total','AUC_death_all_meta']), r2(M.loc['B_ASC_total','p_death_all_meta']))==(0.46,0.77))
check('GLM IgG 1.95 P 0.0041, IgA 1.84 P 0.0025', (r2(G.loc['B_ASC_IgG','OR_glm']), round(G.loc['B_ASC_IgG','p_glm'],4), r2(G.loc['B_ASC_IgA','OR_glm']), round(G.loc['B_ASC_IgA','p_glm'],4))==(1.95,0.0041,1.84,0.0025))
check('T CD8 CM death AUC 0.31 P 0.038; WHO>=6 0.42 P 0.35', (r2(M.loc['T_CD8_CM','AUC_death_all_meta']), round(M.loc['T_CD8_CM','p_death_all_meta'],3), r2(M.loc['T_CD8_CM','AUC_death_WHO6_meta']), r2(M.loc['T_CD8_CM','p_death_WHO6_meta']))==(0.31,0.038,0.42,0.35))
check('CTL COMBAT -0.31 P 0.0067; vdW -0.34 P 0.030', (r2(Cc.loc[('COMBAT 2022','T_CD8_CTL'),'rho']), round(Cc.loc[('COMBAT 2022','T_CD8_CTL'),'p_sev'],4), r2(Cc.loc[('van der Wijst 2021','T_CD8_CTL'),'rho']), round(Cc.loc[('van der Wijst 2021','T_CD8_CTL'),'p_sev'],3))==(-0.31,0.0067,-0.34,0.03))
Mt = 'Public Meta (3 cohorts)'; aucs = [M.loc['HSPC','AUC_death_all_meta'], H.loc[(Mt,'First acute sample'),'AUC_all'], H.loc[(Mt,'Samples > day 7'),'AUC_all']]; ps = [M.loc['HSPC','p_death_all_meta'], H.loc[(Mt,'First acute sample'),'p_all'], H.loc[(Mt,'Samples > day 7'),'p_all']]
check('HSPC overall / >7d AUC 0.68–0.71, P <= 0.011', round(min(aucs),2)==0.68 and round(max(aucs),2)==0.71 and round(max(ps),3)<=0.011, (aucs, ps))
check('NK CD56dim and CD16 mono C1 null', M.loc['NK_CD56dim','p_rho_meta']>0.05 and M.loc['CD16_mono_C1','p_rho_meta']>0.05 and M.loc['NK_CD56dim','p_death_all_meta']>0.05)

## 4. 공개 코호트 독립 재구현 (v8.1 코호트 구성)

In [ ]:
import runpy
_ = runpy.run_path(os.environ.get('PUB_INDEP', 'verify_public_independent.py'), run_name='__main__')

## 요약

In [ ]:
print(f'{len(issues)} 불일치 (원본 ver9 문서·그림 라벨 기준)')
for l in issues: print('-', l)